# GeeksforGeeks Articles — Kaggle Ingestion

This notebook ingests the Kaggle dataset:

`naidukarthi2193/geeks-for-geeks-articles-dataset`

Observed source structure:
- CSV file: `geekstest1.csv`
- 24,489 rows
- Columns: `url`, `title`, `rating`, `content`, `tags`
- The CSV requires a fallback from UTF-8 to `latin-1` encoding.

The notebook keeps only content related to **AI, Data, and Cloud**, performs basic ingestion-level quality checks, standardizes the schema, and saves the result to `data/raw/geeksforgeeks_articles.json`.


## Requirements

Keep dependencies in the project's `requirements.txt`:

```text
pandas
kagglehub
```

No `%pip install` cell is needed in this notebook.


## 1. Imports

In [1]:
from pathlib import Path
import ast
import json
import re

import pandas as pd
import kagglehub

d:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Output path

This notebook assumes it is stored one folder below `notebooks`, for example:

`notebooks/geeks_for_geeks/geeksforgeeks_ingestion.ipynb`

Therefore `../../data/raw` points to the project's shared raw-data folder.


In [2]:
DATA_DIR = Path("../../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = DATA_DIR / "geeksforgeeks_articles.json"

print("Output file:", OUTPUT_FILE.resolve())

Output file: D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw\geeksforgeeks_articles.json


## 3. Download the Kaggle dataset

In [3]:
DATASET_ID = "naidukarthi2193/geeks-for-geeks-articles-dataset"

dataset_path = Path(
    kagglehub.dataset_download(DATASET_ID)
)

print("Dataset downloaded to:")
print(dataset_path)

Dataset downloaded to:
C:\Users\Sarah\.cache\kagglehub\datasets\naidukarthi2193\geeks-for-geeks-articles-dataset\versions\1


## 4. Inspect downloaded files

In [4]:
files = [p for p in dataset_path.rglob("*") if p.is_file()]

print(f"Files found: {len(files)}")

for file in files:
    print("-", file.relative_to(dataset_path))

Files found: 1
- geekstest1.csv


## 5. Load the GeeksforGeeks CSV

The dataset CSV was observed to fail UTF-8 decoding, so the loader automatically falls back to `latin-1`.


In [5]:
def load_gfg_csv(folder):
    csv_files = list(folder.rglob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file found in the GeeksforGeeks dataset."
        )

    frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(
                file,
                encoding="utf-8",
                low_memory=False
            )

        except UnicodeDecodeError:
            print(
                f"UTF-8 failed for {file.name}, "
                "trying latin-1..."
            )

            df = pd.read_csv(
                file,
                encoding="latin-1",
                low_memory=False
            )

        df["_source_file"] = file.name
        frames.append(df)

        print(f"Loaded {file.name}: {df.shape}")
        print("Columns:", df.columns.tolist())

    return pd.concat(
        frames,
        ignore_index=True,
        sort=False
    )


gfg_raw = load_gfg_csv(dataset_path)

print("\nTotal shape:", gfg_raw.shape)

UTF-8 failed for geekstest1.csv, trying latin-1...
Loaded geekstest1.csv: (24489, 6)
Columns: ['url', 'title', 'rating', 'content', 'tags', '_source_file']

Total shape: (24489, 6)


## 6. Preview the raw data

In [6]:
display(gfg_raw.head())

print("\nData types:")
print(gfg_raw.dtypes)

print("\nMissing values:")
print(gfg_raw.isna().sum())

,url,title,rating,content,tags,_source_file
0,https://www.geeksforgeeks.org/python-threshold...,Python | Thresholding techniques using OpenCV ...,0.0,Prerequisite: Simple Thresholding using OpenC...,"{'Python', 'OpenCV', 'Image-Processing'}",geekstest1.csv
1,https://www.geeksforgeeks.org/generation-n-num...,Generation of n numbers with given set of factors,0.0,"Given an array of k numbers factor[], the task...","{'prime-factor', 'Mathematical'}",geekstest1.csv
2,https://www.geeksforgeeks.org/c-operators-ques...,C | Operators | Question 17,1.5,Which of the following can have different mean...,"{'C', 'C-Operators', 'Operators', 'C Quiz'}",geekstest1.csv
3,https://www.geeksforgeeks.org/sympy-subset-siz...,SymPy | Subset.size() in Python,0.0,Subset.size() : size() is a sympy Python libra...,{'Python'},geekstest1.csv
4,https://www.geeksforgeeks.org/python-pil-image...,Python PIL | ImageFont.truetype(),0.0,PIL is the Python Imaging Library which provid...,"{'Python', 'Python-pil'}",geekstest1.csv



Data types:
url                 str
title               str
rating          float64
content             str
tags                str
_source_file        str
dtype: object

Missing values:
url             0
title           0
rating          0
content         0
tags            0
_source_file    0
dtype: int64


## 7. Normalize tags

Some records may represent tags as a Python-list-like string while others may already be list-like. This function gives the filtering logic a consistent list representation.


In [7]:
def parse_tags(tags):
    if tags is None:
        return []

    if isinstance(tags, list):
        return [str(tag).strip() for tag in tags if str(tag).strip()]

    try:
        if pd.isna(tags):
            return []
    except (TypeError, ValueError):
        pass

    if isinstance(tags, str):
        text = tags.strip()

        if not text:
            return []

        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, (list, tuple, set)):
                return [
                    str(tag).strip()
                    for tag in parsed
                    if str(tag).strip()
                ]
        except (ValueError, SyntaxError):
            pass

        # Handle simple comma-separated tags when they are not a list string.
        if "," in text:
            return [
                tag.strip()
                for tag in text.split(",")
                if tag.strip()
            ]

        return [text]

    return [str(tags).strip()]

## 8. Check tag normalization

In [8]:
example_tags = gfg_raw.iloc[0]["tags"]

print("Original:")
print(example_tags)
print("Original type:", type(example_tags))

parsed_tags = parse_tags(example_tags)

print("\nParsed:")
print(parsed_tags)
print("Parsed type:", type(parsed_tags))

Original:
{'Python', 'OpenCV', 'Image-Processing'}
Original type: <class 'str'>

Parsed:
['Image-Processing', 'OpenCV', 'Python']
Parsed type: <class 'list'>


## 9. Define AI, Data, and Cloud keywords

The filter avoids overly broad standalone words such as `data`, `cloud`, and `model` because they can create many false positives.


In [9]:
TOPIC_KEYWORDS = {
    "AI": [
        "artificial intelligence",
        "machine learning",
        "deep learning",
        "generative ai",
        "genai",
        "large language model",
        "large language models",
        "llm",
        "llms",
        "natural language processing",
        "nlp",
        "computer vision",
        "neural network",
        "neural networks",
        "tensorflow",
        "pytorch",
        "scikit-learn"
    ],

    "Data": [
        "data science",
        "data scientist",
        "data engineering",
        "data engineer",
        "data analytics",
        "data analysis",
        "big data",
        "data pipeline",
        "data pipelines",
        "etl",
        "elt",
        "data warehouse",
        "data warehousing",
        "data lake",
        "data lakes",
        "database",
        "databases",
        "sql",
        "postgresql",
        "mysql",
        "mongodb",
        "spark",
        "apache spark",
        "airflow",
        "apache airflow",
        "dbt",
        "pandas",
        "numpy"
    ],

    "Cloud": [
        "cloud computing",
        "cloud architecture",
        "cloud infrastructure",
        "cloud engineering",
        "cloud engineer",
        "aws",
        "amazon web services",
        "azure",
        "microsoft azure",
        "google cloud",
        "google cloud platform",
        "gcp",
        "serverless",
        "kubernetes",
        "docker",
        "cloud native"
    ]
}

## 10. Topic classification helpers

In [10]:
def safe_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def classify_topics(text):
    text = safe_text(text).lower()
    matched_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():

        for keyword in keywords:

            # Short technical terms such as AI, SQL, AWS, GCP, ETL
            # should match as complete tokens rather than inside words.
            if (
                len(keyword) <= 4
                and keyword.replace("-", "").isalnum()
            ):
                pattern = rf"(?<!\w){re.escape(keyword)}(?!\w)"

                if re.search(pattern, text):
                    matched_topics.append(topic)
                    break

            elif keyword in text:
                matched_topics.append(topic)
                break

    return matched_topics

## 11. Filter to AI / Data / Cloud

For GeeksforGeeks, classification uses:
- title
- tags
- the beginning of the article content

Tags and title receive the clearest semantic signal, while a limited content sample helps capture relevant articles whose titles are less explicit.


In [11]:
def build_search_text(row):
    tag_list = parse_tags(row.get("tags"))

    tags_text = " ".join(tag_list)

    return " ".join([
        safe_text(row.get("title")),
        tags_text,
        safe_text(row.get("content"))[:5000]
    ])


gfg_filtered = gfg_raw.copy()

gfg_filtered["_topics"] = gfg_filtered.apply(
    lambda row: classify_topics(
        build_search_text(row)
    ),
    axis=1
)

gfg_filtered = gfg_filtered[
    gfg_filtered["_topics"].map(len) > 0
].copy()

print(
    f"GeeksforGeeks: {len(gfg_raw):,} raw records -> "
    f"{len(gfg_filtered):,} AI/Data/Cloud records"
)

GeeksforGeeks: 24,489 raw records -> 2,466 AI/Data/Cloud records


## 12. Check topic distribution

In [12]:
topic_counts = (
    gfg_filtered["_topics"]
    .explode()
    .value_counts()
)

print(topic_counts)

_topics
Data     2013
AI        551
Cloud      11
Name: count, dtype: int64


## 13. Standardize to the project schema

The source provides `url`, `title`, `content`, `tags`, and `rating`.

Your common schema does not currently include `rating`, so it is not included in the standardized output.

The Kaggle source does not provide separate author, publication date, or description fields, so those values are left blank rather than invented.


In [13]:
gfg_standardized = pd.DataFrame({
    "source": ["GeeksforGeeks"] * len(gfg_filtered),

    "category": gfg_filtered["_topics"].apply(
        lambda topics: ", ".join(topics)
    ),

    "title": gfg_filtered["title"]
        .fillna("")
        .astype(str)
        .str.strip(),

    "author": [""] * len(gfg_filtered),

    "publication_date": [""] * len(gfg_filtered),

    "description": [""] * len(gfg_filtered),

    "url": gfg_filtered["url"]
        .fillna("")
        .astype(str)
        .str.strip(),

    "content": gfg_filtered["content"]
        .fillna("")
        .astype(str)
        .str.strip(),

    "tags": gfg_filtered["tags"].apply(parse_tags)
}).reset_index(drop=True)

display(gfg_standardized.head())

,source,category,title,author,publication_date,description,url,content,tags
0,GeeksforGeeks,Data,Python | Pandas Series.transform(),,,,https://www.geeksforgeeks.org/python-pandas-se...,Pandas series is a One-dimensional ndarray wit...,"[Python pandas-series-methods, Python, Python-..."
1,GeeksforGeeks,Data,Python | Numpy MaskedArray.__rpow__,,,,https://www.geeksforgeeks.org/python-numpy-mas...,numpy.ma.MaskedArray class is a subclass of nd...,"[Python numpy-ndarray, Python, Python-numpy]"
2,GeeksforGeeks,Data,numpy.not_equal() in Python,,,,https://www.geeksforgeeks.org/numpy-not_equal-...,"About : \nnumpy.not_equal(x1, x2[, out]) : che...","[Python-numpy, Python, Python numpy-Logic Func..."
3,GeeksforGeeks,Data,Python program to count words in a sentence,,,,https://www.geeksforgeeks.org/python-program-t...,Data preprocessing is an important task in tex...,"[Python, Python string-programs, Python Programs]"
4,GeeksforGeeks,AI,Text Preprocessing in Python | Set ? 1,,,,https://www.geeksforgeeks.org/text-preprocessi...,Prerequisites: Introduction to NLPWhenever we ...,"[Machine Learning, Natural-language-processing..."


## 14. Basic ingestion-level data quality

In [14]:
gfg_final = gfg_standardized.copy()

# Remove records with no useful article data.
gfg_final = gfg_final[
    (gfg_final["title"] != "")
    & (gfg_final["content"] != "")
].copy()

# Remove exact duplicate URLs when URLs are available.
with_url = (
    gfg_final[gfg_final["url"] != ""]
    .drop_duplicates(subset=["url"])
)

# For records without URLs, use title + content for deduplication.
without_url = (
    gfg_final[gfg_final["url"] == ""]
    .drop_duplicates(subset=["title", "content"])
)

gfg_final = pd.concat(
    [with_url, without_url],
    ignore_index=True
)

print("Final records:", len(gfg_final))

print("\nMissing values:")
print(gfg_final.isna().sum())

print("\nDuplicate URLs:")
print(
    gfg_final.loc[
        gfg_final["url"] != "",
        "url"
    ].duplicated().sum()
)

Final records: 2454

Missing values:
source              0
category            0
title               0
author              0
publication_date    0
description         0
url                 0
content             0
tags                0
dtype: int64

Duplicate URLs:
0


## 15. Manually inspect filtered records

Keyword filtering should be spot-checked before the output is treated as curated data.


In [15]:
pd.set_option("display.max_colwidth", 100)

display(
    gfg_final[
        ["category", "title", "tags"]
    ].sample(
        min(20, len(gfg_final)),
        random_state=42
    )
)

,category,title,tags
2116,Data,numpy string operations | swapcase() function,"[Python-numpy, Python, Python numpy-String Operation]"
700,Data,Python | Numpy matrix.squeeze(),"[Python-numpy, Python numpy-Matrix Function, Python]"
1165,Data,Creating a Pandas dataframe using list of tuples,"[Python, Technical Scripter 2018, Python pandas-dataFrame, pandas-dataframe-program, Technical S..."
2418,Data,Python | Pandas Series.hasnans,"[Python pandas-series-methods, Python, Python-pandas, Python pandas-series]"
1627,Data,SQL | NOT Operator,"[SQL, SQL-Clauses-Operators]"
321,Data,MAQ Software Interview Experience | Set 5,"[Sorting, Interview Experiences, Linked List, Merge Sort, MAQ Software]"
1258,Data,Python | Pandas dataframe.rmod(),"[Python pandas-dataFrame-methods, Python, Python pandas-dataFrame, Python-pandas]"
495,AI,Python | Image Classification using keras,"[Machine Learning, Python]"
507,AI,NLP | Creating Shallow Tree,"[Machine Learning, Natural-language-processing, Python, Python-nltk]"
610,Data,Python | Pandas Dataframe.sample(),"[Python pandas-dataFrame-methods, Python, Python pandas-dataFrame, Python-pandas]"


## 16. Save to `data/raw/geeksforgeeks_articles.json`

In [16]:
records = gfg_final.to_dict(
    orient="records"
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Saved {len(records):,} records -> "
    f"{OUTPUT_FILE.resolve()}"
)

Saved 2,454 records -> D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw\geeksforgeeks_articles.json


## 17. Verify the saved JSON

In [17]:
with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    saved_records = json.load(f)

print("Saved records:", len(saved_records))

if saved_records:
    print("\nFirst record:")
    print(
        json.dumps(
            saved_records[0],
            ensure_ascii=False,
            indent=2
        )[:2500]
    )

Saved records: 2454

First record:
{
  "source": "GeeksforGeeks",
  "category": "Data",
  "title": "Python | Pandas Series.transform()",
  "author": "",
  "publication_date": "",
  "description": "",
  "url": "https://www.geeksforgeeks.org/python-pandas-series-transform/",
  "content": "Pandas series is a One-dimensional ndarray with axis labels. The labels need not be unique but must be a hashable type. The object supports both integer- and label-based indexing and provides a host of methods for performing operations involving the index.Pandas Series.transform() function Call func (the passed function) on self producing a Series with transformed values and that has the same axis length as self.Syntax: Series.transform(func, axis=0, *args, **kwargs)Parameter : \nfunc :  If a function, must either work when passed a Series or when passed to Series.apply\naxis :  Parameter needed for compatibility with DataFrame.\n*args :  Positional arguments to pass to func.\n**kwargs :   Keyword argum

## Output

This ingestion notebook produces:

`data/raw/geeksforgeeks_articles.json`

The output uses the same standardized fields as the other project sources:

`source`, `category`, `title`, `author`, `publication_date`, `description`, `url`, `content`, `tags`

Further profiling, validation, transformations, and integration should remain in the later pipeline stages rather than being mixed into source ingestion.
